In [1]:
# import libraries
import polars as pl
from datetime import date
from tqdm.notebook import tqdm
import plotly.express as px

pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_cols(-1)
pl.Config.set_fmt_str_lengths(100)
pl.Config.set_tbl_width_chars(200)

# data path
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent.parent))

from src.config import RAW_DIR, INTERIM_DIR, PROCESSED_DIR, DOCS_DIR

In [2]:
import ray
from cudf_polars.engine.ray import RayEngine

if not ray.is_initialized():
    ray.init()

engine = RayEngine()

print("Ray:", ray.is_initialized())

2026-09-16 14:30:39,892	INFO worker.py:2015 -- Started a local Ray instance. View the dashboard at http://127.0.0.1:8265 


Ray: True


In [3]:
# load data
lf = pl.scan_csv(
    RAW_DIR / "lending_club/Loan_status_2007-2020Q3.gzip",
    infer_schema_length=10000,
    ignore_errors=True)

# 7. Feature Engineering

**Main Objective**

Transform the selected origination-time features into a model-ready representation while preserving predictive information, interpretability, and leakage control.

## 7.1 Feature Selection

In [4]:
KEEP_FEATURE = ['loan_amnt',
                'term',
                'int_rate',
                'grade',
                'emp_length',
                'home_ownership',
                'annual_inc',
                'verification_status',
                'purpose',
                'addr_state',
                'dti',
                'delinq_2yrs',
                'earliest_cr_line',
                'fico_range_low',
                'inq_last_6mths',
                'mths_since_last_record',
                'pub_rec',
                'revol_util',
                'total_acc',
                'initial_list_status',
                'collections_12_mths_ex_med',
                'mths_since_last_major_derog',
                'application_type',
                'tot_coll_amt',
                'tot_cur_bal',
                'open_act_il',
                'open_il_12m',
                'mths_since_rcnt_il',
                'total_bal_il',
                'il_util',                        # high missing + very weak signal
                'open_rv_24m',
                'max_bal_bc',
                'all_util',
                'inq_fi',
                'total_cu_tl',
                'inq_last_12m',
                'mo_sin_old_il_acct',
                'mo_sin_old_rev_tl_op',
                'mo_sin_rcnt_rev_tl_op',
                'mo_sin_rcnt_tl',
                'mort_acc',
                'mths_since_recent_bc',
                'mths_since_recent_inq',
                'num_accts_ever_120_pd',
                'num_actv_rev_tl',
                'num_il_tl',
                'num_tl_120dpd_2m',
                'num_tl_90g_dpd_24m',
                'num_tl_op_past_12m',
                'pct_tl_nvr_dlq',
                'pub_rec_bankruptcies',
                'total_bc_limit']

TARGET = "loan_status"
PERIOD_COL = "issue_d"
TARGET_ENG = "default"

FILTER_FEATURE = KEEP_FEATURE + [TARGET, PERIOD_COL]

In [5]:
DROP_FEATURE_NEW = []
TEMP_KEEP = ["num_tl_30dpd"] + FILTER_FEATURE

In [6]:
# df = lf.select(FILTER_FEATURE)
df = lf.select(TEMP_KEEP)

In [7]:
len(df.collect_schema())

55

## 7.2 Value Selection (Timeframe and Loan Status)

In [8]:
# filter date
df = (df
        .with_columns(pl.col("issue_d").str.strptime(pl.Date, "%b-%Y", strict=False).alias("issue_d"))  # cast date
        .filter(pl.col("issue_d") >= date(2016, 1, 1))                                                  # filter out data before 2016
    )

In [9]:
# filter loan status
LOAN_STATUS_MAP = {
    "Charged Off" : 1,
    "Default" : 1,
    "Does not meet the credit policy. Status:Charged Off" : 1,
    "Fully Paid" : 0,
    "Does not meet the credit policy. Status:Fully Paid" : 0
}

df = (df
      .filter(pl.col(TARGET).is_in(list(LOAN_STATUS_MAP.keys())))  # filter resolved loan status
      .with_columns(
          pl.col(TARGET)
          .replace(LOAN_STATUS_MAP)                                 # simplify loan status
          .alias(TARGET_ENG)
          .cast(pl.Int8))
)  

In [10]:
df.select(pl.len()).collect(engine=engine)

len
u32
994341


In [11]:
df.head(5).collect(engine=engine)

num_tl_30dpd,loan_amnt,term,int_rate,grade,emp_length,home_ownership,annual_inc,verification_status,purpose,addr_state,dti,delinq_2yrs,earliest_cr_line,fico_range_low,inq_last_6mths,mths_since_last_record,pub_rec,revol_util,total_acc,initial_list_status,collections_12_mths_ex_med,mths_since_last_major_derog,application_type,tot_coll_amt,tot_cur_bal,open_act_il,open_il_12m,mths_since_rcnt_il,total_bal_il,il_util,open_rv_24m,max_bal_bc,all_util,inq_fi,total_cu_tl,inq_last_12m,mo_sin_old_il_acct,mo_sin_old_rev_tl_op,mo_sin_rcnt_rev_tl_op,mo_sin_rcnt_tl,mort_acc,mths_since_recent_bc,mths_since_recent_inq,num_accts_ever_120_pd,num_actv_rev_tl,num_il_tl,num_tl_120dpd_2m,num_tl_90g_dpd_24m,num_tl_op_past_12m,pct_tl_nvr_dlq,pub_rec_bankruptcies,total_bc_limit,loan_status,issue_d,default
str,i64,str,str,str,str,str,f64,str,str,str,f64,i64,str,i64,i64,i64,i64,str,i64,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,str,str,date,i8
"""0""",12000,""" 36 months""",""" 7.97%""","""A""","""10+ years""","""OWN""",42000.0,"""Source Verified""","""debt_consolidation""","""CA""",27.74,0,"""Jun-1996""",715,0,80,1,"""37%""",16,"""w""",0,null,"""Individual""","""0""","""30502""","""2""","""1""","""8""","""19045""","""73""","""4""","""7117""","""53""","""1""","""1""","""2""","""131""","""255""","""1""","""1""","""0""","""14""","""8""","""0""","""6""","""7""","""0""","""0""","""3""","""100""",1,"""15500""","""Fully Paid""",2017-09-01,0
"""0""",10000,""" 36 months""",""" 9.44%""","""B""","""3 years""","""MORTGAGE""",55000.0,"""Not Verified""","""debt_consolidation""","""OR""",18.79,0,"""Sep-2005""",695,0,99,1,"""57.5%""",10,"""w""",0,null,"""Individual""","""0""","""340607""","""4""","""1""","""10""","""209187""","""75""","""0""","""6847""","""68""","""1""","""1""","""1""","""144""","""73""","""49""","""10""","""2""","""49""","""8""","""0""","""2""","""5""","""0""","""0""","""1""","""100""",1,"""10500""","""Fully Paid""",2017-09-01,0
"""0""",8000,""" 36 months""",""" 16.02%""","""C""","""< 1 year""","""MORTGAGE""",120000.0,"""Not Verified""","""debt_consolidation""","""NY""",20.36,0,"""Sep-1994""",700,1,null,0,"""92.3%""",34,"""w""",0,null,"""Joint App""","""0""","""388595""","""3""","""2""","""6""","""83572""","""85""","""0""","""0""","""88""","""1""","""24""","""2""","""137""","""276""","""34""","""6""","""6""","""152""","""3""","""0""","""3""","""19""","""0""","""0""","""3""","""97.1""",0,"""0""","""Fully Paid""",2017-09-01,0
"""0""",12800,""" 36 months""",""" 13.59%""","""C""","""5 years""","""RENT""",90000.0,"""Not Verified""","""debt_consolidation""","""TX""",22.63,0,"""Dec-1988""",660,2,null,0,"""86%""",23,"""w""",0,"""94""","""Individual""","""0""","""93375""","""3""","""2""","""6""","""80715""","""91""","""3""","""3777""","""86""","""0""","""1""","""2""","""154""","""345""","""5""","""5""","""0""","""5""","""5""","""0""","""6""","""15""",null,"""0""","""3""","""83""",0,"""14750""","""Fully Paid""",2017-09-01,0
"""0""",15000,""" 36 months""",""" 13.59%""","""C""","""4 years""","""MORTGAGE""",180000.0,"""Source Verified""","""medical""","""TX""",38.07,0,"""Dec-1976""",680,0,null,0,"""66.6%""",50,"""w""",0,null,"""Individual""","""0""","""682000""","""4""","""0""","""13""","""190233""","""79""","""1""","""22127""","""68""","""2""","""8""","""0""","""132""","""489""","""16""","""13""","""4""","""16""","""13""","""0""","""14""","""17""","""0""","""0""","""0""","""100""",0,"""116700""","""Fully Paid""",2017-09-01,0


## 7.3 Data Normalization and Casting

In [12]:
# cast data type
df = df.with_columns([

    # cast term to int
    pl.col("term")
      .str.strip_chars()
      .str.replace(" months", "")
      .cast(pl.Int8, strict=False),

    # cast int_rate
    pl.col("int_rate")
      .str.strip_chars()
      .str.replace("%", "")
      .cast(pl.Float64, strict=False),

    # cast revol_util
    pl.col("revol_util")
      .str.strip_chars()
      .str.replace("%", "")
      .cast(pl.Float64, strict=False),

    # cast emp_length
    pl.col("emp_length")
      .str.strip_chars()
      .str.replace(" years", "")
      .str.replace(" year", "")
      .str.replace("< 1", "0")
      .str.replace("10+", "10")
      .cast(pl.Int8, strict=False)
])

In [13]:
CAST_FEATURE ={'loan_amnt': pl.Int64,
                'term': pl.Int64,
                'int_rate': pl.Float64,
                'grade': pl.Categorical,
                'emp_length': pl.Int64,
                'home_ownership': pl.Categorical,
                'annual_inc': pl.Float64,
                'verification_status': pl.Categorical,
                'issue_d': pl.Date,
                'loan_status': pl.Categorical,
                'purpose': pl.Categorical,
                'addr_state': pl.Categorical,
                'dti': pl.Float64,
                'delinq_2yrs': pl.Int64,
                'earliest_cr_line': pl.Date,
                'fico_range_low': pl.Int64,
                'inq_last_6mths': pl.Int64,
                'mths_since_last_record': pl.Int64,
                'pub_rec': pl.Int64,
                'revol_util': pl.Float64,
                'total_acc': pl.Int64,
                'initial_list_status': pl.Categorical,
                'collections_12_mths_ex_med': pl.Int64,
                'mths_since_last_major_derog': pl.Int64,
                'application_type': pl.Categorical,
                'tot_coll_amt': pl.Float64,
                'tot_cur_bal': pl.Float64,
                'open_act_il': pl.Int64,
                'open_il_12m': pl.Int64,
                'mths_since_rcnt_il': pl.Int64,
                'total_bal_il': pl.Float64,
                'il_util': pl.Float64,
                'open_rv_24m': pl.Int64,
                'max_bal_bc': pl.Float64,
                'all_util': pl.Float64,
                'inq_fi': pl.Int64,
                'total_cu_tl': pl.Int64,
                'inq_last_12m': pl.Int64,
                'mo_sin_old_il_acct': pl.Int64,
                'mo_sin_old_rev_tl_op': pl.Int64,
                'mo_sin_rcnt_rev_tl_op': pl.Int64,
                'mo_sin_rcnt_tl': pl.Int64,
                'mort_acc': pl.Int64,
                'mths_since_recent_bc': pl.Int64,
                'mths_since_recent_inq': pl.Int64,
                'num_accts_ever_120_pd': pl.Int64,
                'num_actv_rev_tl': pl.Int64,
                'num_il_tl': pl.Int64,
                'num_tl_120dpd_2m': pl.Int64,
                'num_tl_90g_dpd_24m': pl.Int64,
                'num_tl_op_past_12m': pl.Int64,
                'pct_tl_nvr_dlq': pl.Float64,
                'pub_rec_bankruptcies': pl.Int64,
                'total_bc_limit': pl.Float64,

                'num_tl_30dpd' : pl.Int64}              # temporary

In [14]:
# cast and cast earliest credit
cast_exec = [pl.col(column).cast(dtype, strict=False)
            for column, dtype in CAST_FEATURE.items()
            if column != "earliest_cr_line"]

cast_exec = cast_exec + [pl.col("earliest_cr_line").str.strptime(pl.Date, "%b-%Y", strict=False)]

df = df.with_columns(cast_exec)

In [16]:
df.head(5).collect(engine=engine)

num_tl_30dpd,loan_amnt,term,int_rate,grade,emp_length,home_ownership,annual_inc,verification_status,purpose,addr_state,dti,delinq_2yrs,earliest_cr_line,fico_range_low,inq_last_6mths,mths_since_last_record,pub_rec,revol_util,total_acc,initial_list_status,collections_12_mths_ex_med,mths_since_last_major_derog,application_type,tot_coll_amt,tot_cur_bal,open_act_il,open_il_12m,mths_since_rcnt_il,total_bal_il,il_util,open_rv_24m,max_bal_bc,all_util,inq_fi,total_cu_tl,inq_last_12m,mo_sin_old_il_acct,mo_sin_old_rev_tl_op,mo_sin_rcnt_rev_tl_op,mo_sin_rcnt_tl,mort_acc,mths_since_recent_bc,mths_since_recent_inq,num_accts_ever_120_pd,num_actv_rev_tl,num_il_tl,num_tl_120dpd_2m,num_tl_90g_dpd_24m,num_tl_op_past_12m,pct_tl_nvr_dlq,pub_rec_bankruptcies,total_bc_limit,loan_status,issue_d,default
i64,i64,i64,f64,cat,i64,cat,f64,cat,cat,cat,f64,i64,date,i64,i64,i64,i64,f64,i64,cat,i64,i64,cat,f64,f64,i64,i64,i64,f64,f64,i64,f64,f64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,f64,i64,f64,cat,date,i8
0,12000,36,7.97,"""A""",null,"""OWN""",42000.0,"""Source Verified""","""debt_consolidation""","""CA""",27.74,0,1996-06-01,715,0,80,1,37.0,16,"""w""",0,null,"""Individual""",0.0,30502.0,2,1,8,19045.0,73.0,4,7117.0,53.0,1,1,2,131,255,1,1,0,14,8,0,6,7,0,0,3,100.0,1,15500.0,"""Fully Paid""",2017-09-01,0
0,10000,36,9.44,"""B""",3,"""MORTGAGE""",55000.0,"""Not Verified""","""debt_consolidation""","""OR""",18.79,0,2005-09-01,695,0,99,1,57.5,10,"""w""",0,null,"""Individual""",0.0,340607.0,4,1,10,209187.0,75.0,0,6847.0,68.0,1,1,1,144,73,49,10,2,49,8,0,2,5,0,0,1,100.0,1,10500.0,"""Fully Paid""",2017-09-01,0
0,8000,36,16.02,"""C""",0,"""MORTGAGE""",120000.0,"""Not Verified""","""debt_consolidation""","""NY""",20.36,0,1994-09-01,700,1,null,0,92.3,34,"""w""",0,null,"""Joint App""",0.0,388595.0,3,2,6,83572.0,85.0,0,0.0,88.0,1,24,2,137,276,34,6,6,152,3,0,3,19,0,0,3,97.1,0,0.0,"""Fully Paid""",2017-09-01,0
0,12800,36,13.59,"""C""",5,"""RENT""",90000.0,"""Not Verified""","""debt_consolidation""","""TX""",22.63,0,1988-12-01,660,2,null,0,86.0,23,"""w""",0,94,"""Individual""",0.0,93375.0,3,2,6,80715.0,91.0,3,3777.0,86.0,0,1,2,154,345,5,5,0,5,5,0,6,15,null,0,3,83.0,0,14750.0,"""Fully Paid""",2017-09-01,0
0,15000,36,13.59,"""C""",4,"""MORTGAGE""",180000.0,"""Source Verified""","""medical""","""TX""",38.07,0,1976-12-01,680,0,null,0,66.6,50,"""w""",0,null,"""Individual""",0.0,682000.0,4,0,13,190233.0,79.0,1,22127.0,68.0,2,8,0,132,489,16,13,4,16,13,0,14,17,0,0,0,100.0,0,116700.0,"""Fully Paid""",2017-09-01,0


## 7.4 Missingness

In [21]:
def null_summary(df):
    total_rows = df.select(pl.len()).collect(engine=engine).item()

    null_summary = (
        df
        .select([
            pl.col(c).is_null().sum().alias(c)
            for c in df.collect_schema().names()
        ])
        .unpivot(
            variable_name="feature",
            value_name="null_count",
        )
        .with_columns(
            (pl.col("null_count") / total_rows * 100)
            .round(4)
            .alias("null_pct")
        )
        .filter(pl.col("null_count") > 0)
        .sort("null_count", descending=True)
        .collect()
    )

    print(f"Columns with nulls: {null_summary.height} / {len(df.collect_schema().names())}")
    print(null_summary)

In [22]:
null_summary(df)

Columns with nulls: 22 / 56
shape: (22, 3)
┌─────────────────────────────┬────────────┬──────────┐
│ feature                     ┆ null_count ┆ null_pct │
│ ---                         ┆ ---        ┆ ---      │
│ str                         ┆ u32        ┆ f64      │
╞═════════════════════════════╪════════════╪══════════╡
│ mths_since_last_record      ┆ 819663     ┆ 82.4328  │
│ mths_since_last_major_derog ┆ 725489     ┆ 72.9618  │
│ emp_length                  ┆ 400674     ┆ 40.2954  │
│ il_util                     ┆ 139599     ┆ 14.0393  │
│ mths_since_recent_inq       ┆ 100068     ┆ 10.0638  │
│ num_tl_120dpd_2m            ┆ 41542      ┆ 4.1778   │
│ mths_since_rcnt_il          ┆ 28537      ┆ 2.8699   │
│ mo_sin_old_il_acct          ┆ 28486      ┆ 2.8648   │
│ mths_since_recent_bc        ┆ 11943      ┆ 1.2011   │
│ dti                         ┆ 1107       ┆ 0.1113   │
│ …                           ┆ …          ┆ …        │
│ total_cu_tl                 ┆ 57         ┆ 0.0057   │
│ inq

### 7.4.1 Months Since Feature Related

In [23]:
FEATURE_MTHS_ENG_FILL = ["mths_since_recent_bc", "mths_since_recent_inq", "mths_since_rcnt_il"]
FEATURE_MTHS_ENG_NULL = ["mths_since_recent_inq", "mths_since_last_record", "mths_since_last_major_derog"]

In [24]:
# mths imputation
df = (df
            .with_columns([
                pl.col(col).is_null().cast(pl.Int8).alias(f"{col}_null")
                for col in FEATURE_MTHS_ENG_NULL
            ])
            .with_columns([
                pl.col(col).fill_null(0)
                for col in FEATURE_MTHS_ENG_FILL
            ])
            .select(pl.exclude(FEATURE_MTHS_ENG_NULL))
)

In [25]:
df.head().collect()

num_tl_30dpd,loan_amnt,term,int_rate,grade,emp_length,home_ownership,annual_inc,verification_status,purpose,addr_state,dti,delinq_2yrs,earliest_cr_line,fico_range_low,inq_last_6mths,pub_rec,revol_util,total_acc,initial_list_status,collections_12_mths_ex_med,application_type,tot_coll_amt,tot_cur_bal,open_act_il,open_il_12m,mths_since_rcnt_il,total_bal_il,il_util,open_rv_24m,max_bal_bc,all_util,inq_fi,total_cu_tl,inq_last_12m,mo_sin_old_il_acct,mo_sin_old_rev_tl_op,mo_sin_rcnt_rev_tl_op,mo_sin_rcnt_tl,mort_acc,mths_since_recent_bc,num_accts_ever_120_pd,num_actv_rev_tl,num_il_tl,num_tl_120dpd_2m,num_tl_90g_dpd_24m,num_tl_op_past_12m,pct_tl_nvr_dlq,pub_rec_bankruptcies,total_bc_limit,loan_status,issue_d,default,mths_since_recent_inq_null,mths_since_last_record_null,mths_since_last_major_derog_null
i64,i64,i64,f64,cat,i64,cat,f64,cat,cat,cat,f64,i64,date,i64,i64,i64,f64,i64,cat,i64,cat,f64,f64,i64,i64,i64,f64,f64,i64,f64,f64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,f64,i64,f64,cat,date,i8,i8,i8,i8
0,12000,36,7.97,"""A""",null,"""OWN""",42000.0,"""Source Verified""","""debt_consolidation""","""CA""",27.74,0,1996-06-01,715,0,1,37.0,16,"""w""",0,"""Individual""",0.0,30502.0,2,1,8,19045.0,73.0,4,7117.0,53.0,1,1,2,131,255,1,1,0,14,0,6,7,0,0,3,100.0,1,15500.0,"""Fully Paid""",2017-09-01,0,0,0,1
0,10000,36,9.44,"""B""",3,"""MORTGAGE""",55000.0,"""Not Verified""","""debt_consolidation""","""OR""",18.79,0,2005-09-01,695,0,1,57.5,10,"""w""",0,"""Individual""",0.0,340607.0,4,1,10,209187.0,75.0,0,6847.0,68.0,1,1,1,144,73,49,10,2,49,0,2,5,0,0,1,100.0,1,10500.0,"""Fully Paid""",2017-09-01,0,0,0,1
0,8000,36,16.02,"""C""",0,"""MORTGAGE""",120000.0,"""Not Verified""","""debt_consolidation""","""NY""",20.36,0,1994-09-01,700,1,0,92.3,34,"""w""",0,"""Joint App""",0.0,388595.0,3,2,6,83572.0,85.0,0,0.0,88.0,1,24,2,137,276,34,6,6,152,0,3,19,0,0,3,97.1,0,0.0,"""Fully Paid""",2017-09-01,0,0,1,1
0,12800,36,13.59,"""C""",5,"""RENT""",90000.0,"""Not Verified""","""debt_consolidation""","""TX""",22.63,0,1988-12-01,660,2,0,86.0,23,"""w""",0,"""Individual""",0.0,93375.0,3,2,6,80715.0,91.0,3,3777.0,86.0,0,1,2,154,345,5,5,0,5,0,6,15,null,0,3,83.0,0,14750.0,"""Fully Paid""",2017-09-01,0,0,1,0
0,15000,36,13.59,"""C""",4,"""MORTGAGE""",180000.0,"""Source Verified""","""medical""","""TX""",38.07,0,1976-12-01,680,0,0,66.6,50,"""w""",0,"""Individual""",0.0,682000.0,4,0,13,190233.0,79.0,1,22127.0,68.0,2,8,0,132,489,16,13,4,16,0,14,17,0,0,0,100.0,0,116700.0,"""Fully Paid""",2017-09-01,0,0,1,1


In [26]:
null_summary(df)

Columns with nulls: 17 / 56
shape: (17, 3)
┌────────────────────┬────────────┬──────────┐
│ feature            ┆ null_count ┆ null_pct │
│ ---                ┆ ---        ┆ ---      │
│ str                ┆ u32        ┆ f64      │
╞════════════════════╪════════════╪══════════╡
│ emp_length         ┆ 400674     ┆ 40.2954  │
│ il_util            ┆ 139599     ┆ 14.0393  │
│ num_tl_120dpd_2m   ┆ 41542      ┆ 4.1778   │
│ mo_sin_old_il_acct ┆ 28486      ┆ 2.8648   │
│ dti                ┆ 1107       ┆ 0.1113   │
│ revol_util         ┆ 943        ┆ 0.0948   │
│ all_util           ┆ 195        ┆ 0.0196   │
│ total_cu_tl        ┆ 57         ┆ 0.0057   │
│ inq_last_12m       ┆ 57         ┆ 0.0057   │
│ open_act_il        ┆ 56         ┆ 0.0056   │
│ open_il_12m        ┆ 56         ┆ 0.0056   │
│ total_bal_il       ┆ 56         ┆ 0.0056   │
│ open_rv_24m        ┆ 56         ┆ 0.0056   │
│ max_bal_bc         ┆ 56         ┆ 0.0056   │
│ inq_fi             ┆ 56         ┆ 0.0056   │
│ inq_last_6mths 

### 7.4.2 Employment

In [27]:
# employment imputation
df = (df
            .with_columns(
                pl.col("emp_length").is_null().cast(pl.Int8).alias("emp_length_null"),
                pl.col("emp_length").fill_null(0)
            ))

In [28]:
null_summary(df)

Columns with nulls: 16 / 57
shape: (16, 3)
┌────────────────────┬────────────┬──────────┐
│ feature            ┆ null_count ┆ null_pct │
│ ---                ┆ ---        ┆ ---      │
│ str                ┆ u32        ┆ f64      │
╞════════════════════╪════════════╪══════════╡
│ il_util            ┆ 139599     ┆ 14.0393  │
│ num_tl_120dpd_2m   ┆ 41542      ┆ 4.1778   │
│ mo_sin_old_il_acct ┆ 28486      ┆ 2.8648   │
│ dti                ┆ 1107       ┆ 0.1113   │
│ revol_util         ┆ 943        ┆ 0.0948   │
│ all_util           ┆ 195        ┆ 0.0196   │
│ total_cu_tl        ┆ 57         ┆ 0.0057   │
│ inq_last_12m       ┆ 57         ┆ 0.0057   │
│ open_act_il        ┆ 56         ┆ 0.0056   │
│ open_il_12m        ┆ 56         ┆ 0.0056   │
│ total_bal_il       ┆ 56         ┆ 0.0056   │
│ open_rv_24m        ┆ 56         ┆ 0.0056   │
│ max_bal_bc         ┆ 56         ┆ 0.0056   │
│ inq_fi             ┆ 56         ┆ 0.0056   │
│ inq_last_6mths     ┆ 1          ┆ 0.0001   │
│ pct_tl_nvr_dlq 

### 7.4.3 Installment Loan Utilization

In [29]:
feature_score = pl.read_parquet(INTERIM_DIR / "feature_score.parquet")
feature_score.filter(pl.col("feature")=="il_util")

feature,dtype,n_unique,missing_pct,iv,auc,mutual_info,corr_spearman,corr_pb,cramers_v,chi2_pvalue,ks_stat,ks_pvalue,iv_strength,description,category,description_right,category_right
str,str,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str,str,str,str
"""il_util""","""Float64""",267,14.0393,0.0113,0.531,0.0034,0.0436,0.04,null,null,0.0508,6.2713e-316,"""Very weak""","""Ratio of total current balance to high credit/credit limit on all install acct""","""installment_credit""","""Ratio of total current balance to high credit/credit limit on all install acct""","""installment_credit"""


In [30]:
data_profile = pl.read_parquet(INTERIM_DIR / "data_profile_model_base.parquet")
data_profile.filter(pl.col("feature")=="il_util")

feature,description,dtype,count,null_count,null_pct,unique_count,unique_sample,mean,median,mode
str,str,str,i64,i64,f64,i64,str,str,str,str
"""il_util""","""Ratio of total current balance to high credit/credit limit on all install acct""","""Float64""",994341,139599,14.039349,267,"""[0.0, 1.0, 2.0, 3.0, 4.0]""","""70.41906680612395""","""73.0""","""78.0"""


`il_util` was excluded due to relatively high missingness (14.0%) combined with very weak predictive signal. The feature showed low IV (0.0113), near-random univariate AUC (0.531), and weak dependence with the default outcome. Retaining the feature would introduce additional missing-value handling with limited evidence of incremental predictive value.

In [31]:
DROP_FEATURE_NEW.append("il_util")

### 7.4.4 Number of accounts currently 120 days past due (updated in past 2 months)

In [32]:
(df
    .select("num_tl_120dpd_2m", "num_tl_30dpd")
    .filter(pl.col("num_tl_120dpd_2m") <= pl.col("num_tl_30dpd"))
    .group_by("num_tl_30dpd", "num_tl_120dpd_2m")
    .agg(pl.len())

    .collect(engine=engine))

num_tl_30dpd,num_tl_120dpd_2m,len
i64,i64,u32
3,0,10
1,0,2315
0,0,949752
1,1,4
2,0,103
4,0,1


In [33]:
missing_120 = (
    df
    .filter(pl.col("num_tl_120dpd_2m").is_null())
    .select([
        pl.col("num_tl_30dpd").mean().alias("mean_30dpd"),
        pl.col("num_tl_90g_dpd_24m").mean().alias("mean_90dpd_24m"),
        pl.col("num_tl_30dpd").eq(0).mean().alias("pct_30dpd_zero"),
        pl.col("num_tl_90g_dpd_24m").eq(0).mean().alias("pct_90dpd_zero"),
    ])
)

missing_120.collect(engine=engine)

mean_30dpd,mean_90dpd_24m,pct_30dpd_zero,pct_90dpd_zero
f64,f64,f64,f64
0.004189,0.185186,0.995956,0.87781


In [34]:
df.filter(
    pl.col("num_tl_120dpd_2m").is_null()
).group_by([
    "num_tl_30dpd"
]).agg(
    pl.len().alias("count")
).sort("count", descending=True).collect(engine=engine)

num_tl_30dpd,count
i64,u32
0,41374
1,162
2,6


In [35]:
NUM_FEATURE = [i 
               for (i,d) in df.collect_schema().items()
               if d.is_numeric()]


CAT_FETURE = [i
              for i in df.collect_schema().names()
              if i not in NUM_FEATURE]

In [36]:
temp_df = df.with_columns(pl.col("issue_d").dt.year().alias("issue_year"))

In [37]:
(temp_df
    .select(NUM_FEATURE)
    .collect(engine=engine)
    .to_pandas()
    .corr()["num_tl_120dpd_2m"]
    .abs()
    .sort_values(ascending=False))

num_tl_120dpd_2m                    1.000000
num_tl_90g_dpd_24m                  0.064060
delinq_2yrs                         0.042747
mths_since_last_major_derog_null    0.041431
num_accts_ever_120_pd               0.026741
pct_tl_nvr_dlq                      0.017769
fico_range_low                      0.016166
mort_acc                            0.008753
mo_sin_old_rev_tl_op                0.008365
revol_util                          0.007982
tot_cur_bal                         0.007852
total_acc                           0.007314
num_tl_op_past_12m                  0.005863
mo_sin_old_il_acct                  0.005854
open_il_12m                         0.005331
mths_since_rcnt_il                  0.005237
mo_sin_rcnt_tl                      0.005170
pub_rec                             0.004847
max_bal_bc                          0.004811
mths_since_recent_bc                0.004606
dti                                 0.003826
il_util                             0.003566
all_util  

`num_tl_120dpd_2m` was imputed using the median within `issue_d`, with issue year as a fallback. This preserves vintage-specific delinquency patterns while avoiding unreliable inference from weakly related features. The median is robust to outliers and appropriate for this count-based variable.

In [38]:
IMPUTE_COL = "num_tl_120dpd_2m"

# median by issue date
issue_date_median = (
    df
    .group_by("issue_d")
    .agg(
        pl.col(IMPUTE_COL).median().alias("_median")
    )
)

# impute using issue-date median, fallback to 0
df = (
    df
    .join(issue_date_median, on="issue_d", how="left")
    .with_columns(
        pl.coalesce(
            pl.col(IMPUTE_COL),
            pl.col("_median"),
            pl.lit(0),
        ).alias(IMPUTE_COL)
    )
    .drop("_median")
)

In [39]:
null_summary(df)

Columns with nulls: 15 / 57
shape: (15, 3)
┌────────────────────┬────────────┬──────────┐
│ feature            ┆ null_count ┆ null_pct │
│ ---                ┆ ---        ┆ ---      │
│ str                ┆ u32        ┆ f64      │
╞════════════════════╪════════════╪══════════╡
│ il_util            ┆ 139599     ┆ 14.0393  │
│ mo_sin_old_il_acct ┆ 28486      ┆ 2.8648   │
│ dti                ┆ 1107       ┆ 0.1113   │
│ revol_util         ┆ 943        ┆ 0.0948   │
│ all_util           ┆ 195        ┆ 0.0196   │
│ total_cu_tl        ┆ 57         ┆ 0.0057   │
│ inq_last_12m       ┆ 57         ┆ 0.0057   │
│ open_act_il        ┆ 56         ┆ 0.0056   │
│ open_il_12m        ┆ 56         ┆ 0.0056   │
│ total_bal_il       ┆ 56         ┆ 0.0056   │
│ open_rv_24m        ┆ 56         ┆ 0.0056   │
│ max_bal_bc         ┆ 56         ┆ 0.0056   │
│ inq_fi             ┆ 56         ┆ 0.0056   │
│ inq_last_6mths     ┆ 1          ┆ 0.0001   │
│ pct_tl_nvr_dlq     ┆ 1          ┆ 0.0001   │
└────────────────

### 7.4.5 Months since oldest bank installment account opened

In [40]:
(temp_df
    .select(NUM_FEATURE)
    .collect(engine=engine)
    .to_pandas()
    .corr()["mo_sin_old_il_acct"]
    .abs()
    .sort_values(ascending=False))

mo_sin_old_il_acct                  1.000000
num_il_tl                           0.362250
total_acc                           0.345487
mo_sin_old_rev_tl_op                0.207516
total_bal_il                        0.206768
mort_acc                            0.187312
tot_cur_bal                         0.178792
open_act_il                         0.155623
mths_since_last_major_derog_null    0.134477
num_accts_ever_120_pd               0.121202
total_cu_tl                         0.116030
max_bal_bc                          0.110538
pct_tl_nvr_dlq                      0.109827
loan_amnt                           0.107666
emp_length_null                     0.106132
total_bc_limit                      0.081061
delinq_2yrs                         0.079252
revol_util                          0.070808
num_actv_rev_tl                     0.068780
mths_since_rcnt_il                  0.066466
int_rate                            0.063818
all_util                            0.063461
annual_inc

`mo_sin_old_il_acct` was imputed using the median within `issue_d`. Moderate correlations with other features were not considered sufficient for model-based imputation. Vintage-specific median imputation preserves temporal differences while providing a robust treatment for the relatively small proportion of missing values.

In [41]:
IMPUTE_COL = "mo_sin_old_il_acct"

# median by issue date
issue_date_median = (
    df
    .group_by("issue_d")
    .agg(
        pl.col(IMPUTE_COL).median().alias("_median")
    )
)

# impute using issue-date median, fallback to 0
df = (
    df
    .join(issue_date_median, on="issue_d", how="left")
    .with_columns(
        pl.coalesce(
            pl.col(IMPUTE_COL),
            pl.col("_median"),
            pl.lit(0),
        ).alias(IMPUTE_COL)
    )
    .drop("_median")
)

In [42]:
null_summary(df)

Columns with nulls: 14 / 57
shape: (14, 3)
┌────────────────┬────────────┬──────────┐
│ feature        ┆ null_count ┆ null_pct │
│ ---            ┆ ---        ┆ ---      │
│ str            ┆ u32        ┆ f64      │
╞════════════════╪════════════╪══════════╡
│ il_util        ┆ 139599     ┆ 14.0393  │
│ dti            ┆ 1107       ┆ 0.1113   │
│ revol_util     ┆ 943        ┆ 0.0948   │
│ all_util       ┆ 195        ┆ 0.0196   │
│ total_cu_tl    ┆ 57         ┆ 0.0057   │
│ inq_last_12m   ┆ 57         ┆ 0.0057   │
│ open_act_il    ┆ 56         ┆ 0.0056   │
│ open_il_12m    ┆ 56         ┆ 0.0056   │
│ total_bal_il   ┆ 56         ┆ 0.0056   │
│ open_rv_24m    ┆ 56         ┆ 0.0056   │
│ max_bal_bc     ┆ 56         ┆ 0.0056   │
│ inq_fi         ┆ 56         ┆ 0.0056   │
│ inq_last_6mths ┆ 1          ┆ 0.0001   │
│ pct_tl_nvr_dlq ┆ 1          ┆ 0.0001   │
└────────────────┴────────────┴──────────┘


### 7.4.6 DTI

In [43]:
(temp_df
    .select(NUM_FEATURE)
    .collect(engine=engine)
    .to_pandas()
    .corr()["dti"]
    .abs()
    .sort_values(ascending=False))

dti                                 1.000000
total_bal_il                        0.160635
open_act_il                         0.145668
num_actv_rev_tl                     0.144654
all_util                            0.135083
total_acc                           0.133702
num_il_tl                           0.128485
int_rate                            0.126700
mths_since_rcnt_il                  0.122214
revol_util                          0.117268
open_il_12m                         0.102324
total_cu_tl                         0.084579
max_bal_bc                          0.077922
annual_inc                          0.062986
pct_tl_nvr_dlq                      0.056056
inq_fi                              0.054215
default                             0.050186
mo_sin_rcnt_tl                      0.047129
term                                0.042417
loan_amnt                           0.040052
mo_sin_old_il_acct                  0.039208
mo_sin_old_rev_tl_op                0.037405
num_tl_op_

`dti` was imputed using the median within `issue_d`. Given its very low missingness (0.11%), a simple vintage-specific median provides a robust treatment while preserving temporal differences in borrower financial profiles.

In [44]:
IMPUTE_COL = "dti"

# median by issue date
issue_date_median = (
    df
    .group_by("issue_d")
    .agg(
        pl.col(IMPUTE_COL).median().alias("_median")
    )
)

# impute using issue-date median, fallback to 0
df = (
    df
    .join(issue_date_median, on="issue_d", how="left")
    .with_columns(
        pl.coalesce(
            pl.col(IMPUTE_COL),
            pl.col("_median"),
            pl.lit(0),
        ).alias(IMPUTE_COL)
    )
    .drop("_median")
)

In [45]:
null_summary(df)

Columns with nulls: 13 / 57
shape: (13, 3)
┌────────────────┬────────────┬──────────┐
│ feature        ┆ null_count ┆ null_pct │
│ ---            ┆ ---        ┆ ---      │
│ str            ┆ u32        ┆ f64      │
╞════════════════╪════════════╪══════════╡
│ il_util        ┆ 139599     ┆ 14.0393  │
│ revol_util     ┆ 943        ┆ 0.0948   │
│ all_util       ┆ 195        ┆ 0.0196   │
│ total_cu_tl    ┆ 57         ┆ 0.0057   │
│ inq_last_12m   ┆ 57         ┆ 0.0057   │
│ open_act_il    ┆ 56         ┆ 0.0056   │
│ open_il_12m    ┆ 56         ┆ 0.0056   │
│ total_bal_il   ┆ 56         ┆ 0.0056   │
│ open_rv_24m    ┆ 56         ┆ 0.0056   │
│ max_bal_bc     ┆ 56         ┆ 0.0056   │
│ inq_fi         ┆ 56         ┆ 0.0056   │
│ inq_last_6mths ┆ 1          ┆ 0.0001   │
│ pct_tl_nvr_dlq ┆ 1          ┆ 0.0001   │
└────────────────┴────────────┴──────────┘


### 7.4.7 Other Imputation and Future Approach

In [46]:
def impute_missing_by_issue_date(
    df: pl.DataFrame | pl.LazyFrame,
    special_features: set[str] | None = None,
):
    special_features = special_features or set()
    schema = df.collect_schema()

    numeric_types = {
        pl.Int8, pl.Int16, pl.Int32, pl.Int64,
        pl.UInt8, pl.UInt16, pl.UInt32, pl.UInt64,
        pl.Float32, pl.Float64,
    }
    categorical_types = {
        pl.String,
        pl.Categorical,
    }

    # detect columns with nulls in one pass
    null_counts = (
        df
        .select([
            pl.col(col).null_count().alias(col)
            for col in schema.names()
        ])
        .collect()
        .row(0, named=True)
    )

    numeric_features = [
        col for col, cnt in null_counts.items()
        if cnt > 0
        and col != "issue_d"
        and col not in special_features
        and schema[col] in numeric_types
    ]

    categorical_features = [
        col for col, cnt in null_counts.items()
        if cnt > 0
        and col != "issue_d"
        and col not in special_features
        and schema[col] in categorical_types
    ]

    # compute all medians in one group_by
    if numeric_features:
        numeric_stats = (
            df
            .group_by("issue_d")
            .agg([
                pl.col(f).median().alias(f"_impute_{f}")
                for f in numeric_features
            ])
        )

        df = (
            df
            .join(numeric_stats, on="issue_d", how="left")
            .with_columns([
                pl.coalesce(
                    pl.col(f),
                    pl.col(f"_impute_{f}"),
                    pl.lit(0).cast(schema[f]),
                ).alias(f)
                for f in numeric_features
            ])
            .drop([f"_impute_{f}" for f in numeric_features])
        )

    # compute all modes in one group_by
    if categorical_features:
        categorical_stats = (
            df
            .group_by("issue_d")
            .agg([
                pl.col(f).drop_nulls().mode().first().alias(f"_impute_{f}")
                for f in categorical_features
            ])
        )

        df = (
            df
            .join(categorical_stats, on="issue_d", how="left")
            .with_columns([
                pl.coalesce(
                    pl.col(f),
                    pl.col(f"_impute_{f}"),
                    pl.lit("Missing"),
                ).alias(f)
                for f in categorical_features
            ])
            .drop([f"_impute_{f}" for f in categorical_features])
        )

    return df

In [47]:
df = impute_missing_by_issue_date(df)

In [48]:
df.sink_parquet(PROCESSED_DIR / "imputed.parquet", engine=engine)

In [49]:
null_summary(df)

Columns with nulls: 0 / 57
shape: (0, 3)
┌─────────┬────────────┬──────────┐
│ feature ┆ null_count ┆ null_pct │
│ ---     ┆ ---        ┆ ---      │
│ str     ┆ u32        ┆ f64      │
╞═════════╪════════════╪══════════╡
└─────────┴────────────┴──────────┘


## 7.5 Transformation

In [50]:
df =  (df.with_columns(
    ((pl.col("issue_d") - pl.col("earliest_cr_line")).dt.total_days() // 30)
    .alias("cr_age_mths")))

In [51]:
df.head().collect(engine=engine)

num_tl_30dpd,loan_amnt,term,int_rate,grade,emp_length,home_ownership,annual_inc,verification_status,purpose,addr_state,dti,delinq_2yrs,earliest_cr_line,fico_range_low,inq_last_6mths,pub_rec,revol_util,total_acc,initial_list_status,collections_12_mths_ex_med,application_type,tot_coll_amt,tot_cur_bal,open_act_il,open_il_12m,mths_since_rcnt_il,total_bal_il,il_util,open_rv_24m,max_bal_bc,all_util,inq_fi,total_cu_tl,inq_last_12m,mo_sin_old_il_acct,mo_sin_old_rev_tl_op,mo_sin_rcnt_rev_tl_op,mo_sin_rcnt_tl,mort_acc,mths_since_recent_bc,num_accts_ever_120_pd,num_actv_rev_tl,num_il_tl,num_tl_120dpd_2m,num_tl_90g_dpd_24m,num_tl_op_past_12m,pct_tl_nvr_dlq,pub_rec_bankruptcies,total_bc_limit,loan_status,issue_d,default,mths_since_recent_inq_null,mths_since_last_record_null,mths_since_last_major_derog_null,emp_length_null,cr_age_mths
i64,i64,i64,f64,cat,i64,cat,f64,cat,cat,cat,f64,i64,date,i64,f64,i64,f64,i64,cat,i64,cat,f64,f64,f64,f64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,i64,i64,i64,i64,i64,i64,i64,f64,i64,i64,f64,i64,f64,cat,date,i8,i8,i8,i8,i8,i64
0,12000,36,7.97,"""A""",0,"""OWN""",42000.0,"""Source Verified""","""debt_consolidation""","""CA""",27.74,0,1996-06-01,715,0.0,1,37.0,16,"""w""",0,"""Individual""",0.0,30502.0,2.0,1.0,8,19045.0,73.0,4.0,7117.0,53.0,1.0,1.0,2.0,131.0,255,1,1,0,14,0,6,7,0.0,0,3,100.0,1,15500.0,"""Fully Paid""",2017-09-01,0,0,0,1,1,258
0,10000,36,9.44,"""B""",3,"""MORTGAGE""",55000.0,"""Not Verified""","""debt_consolidation""","""OR""",18.79,0,2005-09-01,695,0.0,1,57.5,10,"""w""",0,"""Individual""",0.0,340607.0,4.0,1.0,10,209187.0,75.0,0.0,6847.0,68.0,1.0,1.0,1.0,144.0,73,49,10,2,49,0,2,5,0.0,0,1,100.0,1,10500.0,"""Fully Paid""",2017-09-01,0,0,0,1,0,146
0,8000,36,16.02,"""C""",0,"""MORTGAGE""",120000.0,"""Not Verified""","""debt_consolidation""","""NY""",20.36,0,1994-09-01,700,1.0,0,92.3,34,"""w""",0,"""Joint App""",0.0,388595.0,3.0,2.0,6,83572.0,85.0,0.0,0.0,88.0,1.0,24.0,2.0,137.0,276,34,6,6,152,0,3,19,0.0,0,3,97.1,0,0.0,"""Fully Paid""",2017-09-01,0,0,1,1,0,280
0,12800,36,13.59,"""C""",5,"""RENT""",90000.0,"""Not Verified""","""debt_consolidation""","""TX""",22.63,0,1988-12-01,660,2.0,0,86.0,23,"""w""",0,"""Individual""",0.0,93375.0,3.0,2.0,6,80715.0,91.0,3.0,3777.0,86.0,0.0,1.0,2.0,154.0,345,5,5,0,5,0,6,15,0.0,0,3,83.0,0,14750.0,"""Fully Paid""",2017-09-01,0,0,1,0,0,350
0,15000,36,13.59,"""C""",4,"""MORTGAGE""",180000.0,"""Source Verified""","""medical""","""TX""",38.07,0,1976-12-01,680,0.0,0,66.6,50,"""w""",0,"""Individual""",0.0,682000.0,4.0,0.0,13,190233.0,79.0,1.0,22127.0,68.0,2.0,8.0,0.0,132.0,489,16,13,4,16,0,14,17,0.0,0,0,100.0,0,116700.0,"""Fully Paid""",2017-09-01,0,0,1,1,0,496


# Deactivate Engine

In [ ]:
if ray.is_initialized():
    ray.shutdown()
    print("Ray Successfully Shut Down:", not ray.is_initialized())